In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [3]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [4]:
def transform_kor_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [5]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [6]:
kor_dataset = pd.read_csv("../Dataset/KOR_smishing_data_raw/KOR_phishing_data_raw.csv", encoding='utf-8')
kor_dataset.rename(columns={'class': 'label'}, inplace=True)
kor_dataset.head()

,index,content,label
0,1,엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐?,1
1,2,[국외발신]\n[코인원]\n고객님계정이\n해외IP에서\n로그인되였습니다.해외IP차단...,1
2,3,엄마 입금받을수있는\n은행계좌번호\n하나랑 계좌등록\n본인인증땜에\n계좌 4자리\n...,1
3,4,엄마 바빠?나지금\n핸드폰 고장나서 매장에 수리맡기고\n급한대로 예전에\n내명의로 ...,1
4,5,엄마~ 바빠?\n엄마 나 급한 일이 좀 생겨서…\n지금 98만원만 입금해 줄 수 있...,1


In [7]:
# kor_dataset = transform_kor_dict(kor_dataset)
# kor_dataset.head()

In [8]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'kor Dataset_'+'.csv')['URL'].to_list())

In [9]:
kor_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'Korean Dataset_'+'.csv')['URL']
kor_dataset['Message Len'] = [len(i) for i in kor_dataset['content']]
kor_dataset.head()

,index,content,label,Extracted URL,Message Len
0,1,엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐?,1,NaN,49
1,2,[국외발신]\n[코인원]\n고객님계정이\n해외IP에서\n로그인되였습니다.해외IP차단...,1,www.coinonve.com,64
2,3,엄마 입금받을수있는\n은행계좌번호\n하나랑 계좌등록\n본인인증땜에\n계좌 4자리\n...,1,NaN,51
3,4,엄마 바빠?나지금\n핸드폰 고장나서 매장에 수리맡기고\n급한대로 예전에\n내명의로 ...,1,NaN,104
4,5,엄마~ 바빠?\n엄마 나 급한 일이 좀 생겨서…\n지금 98만원만 입금해 줄 수 있...,1,NaN,49


In [10]:
kor_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'Korean Websites Analysis'+'.csv')
kor_website_analysis_data = kor_website_analysis_data.drop(columns=['ham', 'spam'])
kor_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,www.coincneo.com,www.coincneo.com,0,0,-1,0
1,http://tinyurl.com/yfuwrq28,tinyurl.com,13574,0,200,1
2,http://tinyurl.com/y7yxy5gm,tinyurl.com,13574,0,200,1
3,http://fff.kr/auEM,fff.kr,0,0,-1,0
4,https://soo.gd/u053?sco,soo.gd,0,0,-1,0


In [11]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [12]:
kor_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_20808\1880287955.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  kor_website_analysis_data.iloc[0][0]


'www.coincneo.com'

In [13]:
# for row in kor_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [14]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [15]:
extracted_urls = kor_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(kor_website_analysis_data['FQDN'])}
website_data = kor_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [16]:
kor_dataset['FQDN'] = fqdn
kor_dataset['Website Size in KB'] = website_size
kor_dataset['Website Textual Content Length'] = text_content_len
kor_dataset['Status Code'] = status_code
kor_dataset['Parked'] = parked

In [17]:
kor_dataset = kor_dataset.replace('', np.nan)
kor_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_20808\135796785.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  kor_dataset = kor_dataset.replace('', np.nan)


,index,content,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,1,엄마 나 폰이 망가져서 수리맡기고 컴퓨터 문자나라로 메시지 보내고 있어 엄마 지금 바뻐?,1,NaN,49,NaN,NaN,NaN,NaN,NaN
1,2,[국외발신]\n[코인원]\n고객님계정이\n해외IP에서\n로그인되였습니다.해외IP차단...,1,www.coinonve.com,64,www.coinonve.com,0.0,0.0,-1.0,0.0
2,3,엄마 입금받을수있는\n은행계좌번호\n하나랑 계좌등록\n본인인증땜에\n계좌 4자리\n...,1,NaN,51,NaN,NaN,NaN,NaN,NaN
3,4,엄마 바빠?나지금\n핸드폰 고장나서 매장에 수리맡기고\n급한대로 예전에\n내명의로 ...,1,NaN,104,NaN,NaN,NaN,NaN,NaN
4,5,엄마~ 바빠?\n엄마 나 급한 일이 좀 생겨서…\n지금 98만원만 입금해 줄 수 있...,1,NaN,49,NaN,NaN,NaN,NaN,NaN


In [18]:
# Counter(kor_dataset['FQDN'].to_list())
kor_dataset[(kor_dataset['Extracted URL'].notna()) & (kor_dataset['FQDN'].isna())]

,index,content,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [19]:
# Counter(kor_dataset['FQDN'].to_list())
kor_dataset[(kor_dataset['Extracted URL'].notna()) & (kor_dataset['FQDN'].notna())]

,index,content,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
1,2,[국외발신]\n[코인원]\n고객님계정이\n해외IP에서\n로그인되였습니다.해외IP차단...,1,www.coinonve.com,64,www.coinonve.com,0.0,0.0,-1.0,0.0
10,11,https://\nplay.google.com/\nstore/apps/\ndetai...,1,https://play.google.com/store/apps/details?id=...,148,play.google.com,1758015.0,3301.0,200.0,0.0
19,20,아빠 나 폰액정이 나가서\n수리 맡겼어 이번호가\n임시사용하는거라 문자만\n가능해 ...,1,https://play.google.com,316,play.google.com,1758015.0,3301.0,200.0,0.0
28,29,[질병관리청COOV]코로나 19\n전자예방접종증명서 발송\n완료 https://ya...,1,https://ya.mba/3PZ,52,ya.mba,1739.0,123.0,200.0,0.0
29,30,[Web발신]\n여분백신 화이자 예약안내\n코르나 19 백신 접종 예약을\n확인하세...,1,https://url.kr/fn8ocq,66,url.kr,25437.0,1174.0,200.0,0.0
...,...,...,...,...,...,...,...,...,...,...
604,605,(광고)유앤미글로벌 처음해보시는 종목이여도 괜찮습니다. 어느 누가 태어나서 ...,1,www.ynmglobal6.com,272,www.ynmglobal6.com,0.0,0.0,-1.0,0.0
605,606,(광고)유앤미글로벌 장소 제한 없이 어디서나 가능! 투자법에 대해 모르셔도 OK...,1,www.ynmglobal6.com,305,www.ynmglobal6.com,0.0,0.0,-1.0,0.0
1388,1389,■ 프로그램명 : 통합뉴스룸ET\n■ 코너명 : 호모 이코노미쿠스\n■ 방송시간 :...,0,https://news.kbs.co.kr/vod/program.do?bcd=0076...,200,news.kbs.co.kr,783.0,0.0,200.0,1.0
2114,2115,■ 프로그램명 : 통합뉴스룸ET\n■ 코너명 : 호모 이코노미쿠스\n■ 방송시간 :...,0,https://news.kbs.co.kr/vod/program.do?bcd=0076...,200,news.kbs.co.kr,783.0,0.0,200.0,1.0


In [20]:
print(len(kor_dataset))

43209


In [28]:
#messages with URL
print(len(kor_dataset[(kor_dataset['Extracted URL'].notna())]), len(kor_dataset[(kor_dataset['Extracted URL'].notna())])/len(kor_dataset))

127 0.0029392024809646138


In [ ]:
#smish messages with URL
print(len(kor_dataset[(kor_dataset['Extracted URL'].notna()) & (kor_dataset['label']==1)]), len(kor_dataset[(kor_dataset['Extracted URL'].notna()) & (kor_dataset['label']==1)])/len(kor_dataset[kor_dataset['label']==1]))

124 0.2016260162601626


In [24]:
#unique FQDN
len(set(kor_dataset[(kor_dataset['FQDN'].notna())]['FQDN']))

75

In [ ]:
only_unique_live_websites_data = kor_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']==0)]))

#live websites smish
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']==1)]))

30
2
28


In [ ]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']==1)]))

#parked websites smish
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']==1)]))

7
6
6


In [27]:
kor_dataset.to_csv('../Dataset/Refined_Kor_Smishing_Dataset.csv', index=None)